# Let us create meaningful features from the data

In [1]:
import numpy as np
import pandas as pd

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..")))

from src.config import FLIGHT_DATA

import pandas as pd

df = pd.read_csv(FLIGHT_DATA)

In [3]:
df

,id,airline,flight,source,departure,stops,arrival,destination,class,duration,days_left,price
0,0,Vistara,UK-930,Mumbai,Early_Morning,one,Night,Chennai,Business,NaN,40.0,64173
1,1,Air_India,AI-539,Chennai,Evening,one,Morning,Mumbai,Economy,16.08,26.0,4357
2,2,SpiceJet,SG-8107,Delhi,Early_Morning,zero,Morning,Chennai,Economy,2.92,25.0,3251
3,3,NaN,0.00E+00,Hyderabad,Early_Morning,zero,Morning,Bangalore,Economy,1.50,22.0,1776
4,4,Air_India,AI-569,Chennai,Early_Morning,one,Morning,Bangalore,Economy,4.83,20.0,3584
...,...,...,...,...,...,...,...,...,...,...,...,...
39995,39995,Vistara,UK-940,Mumbai,NaN,one,Evening,Bangalore,Economy,21.25,43.0,6450
39996,39996,Vistara,UK-720,Kolkata,Early_Morning,one,Night,Mumbai,Business,14.08,12.0,64831
39997,39997,NaN,UK-874,Hyderabad,Morning,one,Night,Bangalore,Economy,14.33,NaN,8062
39998,39998,Vistara,UK-940,Mumbai,NaN,one,Night,Bangalore,Business,24.67,4.0,84557


# Cleaning

In [15]:

df.loc[ ~df["flight"].str.match(r"[A-Z 0-9]{2}-[0-9]{2,4}",na=False),"flight"].unique()

array(['0.00E+00', '6.00E-219', '6.00E-282', '6.00E-161', '6.00E-126',
       '6.00E-146', '6.00E-139', '6.00E-128', '0.00E-02', '6.00E-135',
       '6.00E-136', '6.00E-113', '6.00E-149', '6.00E-232', '6.00E-234',
       '6.00E-181', '6.00E-132', '6.00E-269', '6.00E-285', '6.00E-218',
       '6.00E-157', '6.00E-201', '6.00E-207', '6.00E-168', '6.00E-295',
       '6.00E-152', '6.00E-276', '6.00E-186', '6.00E-177', '6.00E-154',
       '6.00E-156', '6.00E-121', '6.00E-176', '6.00E-255', '6.00E-171',
       '6.00E-246', '6.00E-236', '6.00E-102', '6.00E-138', '6.00E-183',
       '6.00E-179', '6.00E-296', '6.00E-193', '6.00E-164', '6.00E-294',
       '6.00E-292', '6.00E-227', '6.00E-134', '6.00E-307', '6.00E-165',
       '6.00E-248', '6.00E-198', '6.00E-283', '6.00E-188', '6.00E-205',
       '6.00E-131', '6.00E-286', '6.00E-224', '6.00E-271', '6.00E-184',
       '6.00E-153', '6.00E-289', '6.00E-244', '6.00E-221', '6.00E-308',
       '6.00E-192', '6.00E-268', '6.00E-298', '6.00E-151', '6.00E-

In [17]:
# df[df['flight']=="0.00E+00"]
df[df['airline']=="Indigo"]

,id,airline,flight,source,departure,stops,arrival,destination,class,duration,days_left,price
7,7,Indigo,0.00E+00,Chennai,Evening,one,Night,Hyderabad,Economy,4.42,38.0,6880
8,8,Indigo,0.00E+00,Kolkata,Morning,one,Evening,Mumbai,Economy,4.67,35.0,6521
17,17,Indigo,0.00E+00,Hyderabad,NaN,one,Night,Chennai,Economy,8.25,22.0,5220
34,34,Indigo,0.00E+00,Chennai,Early_Morning,one,Evening,Mumbai,Economy,NaN,45.0,2792
42,42,Indigo,6.00E-219,Mumbai,Afternoon,one,Night,Kolkata,Economy,6.33,NaN,10301
...,...,...,...,...,...,...,...,...,...,...,...,...
39952,39952,Indigo,0.00E+00,Delhi,Afternoon,one,Evening,Chennai,Economy,7.42,11.0,7950
39964,39964,Indigo,0.00E+00,Kolkata,Evening,one,Early_Morning,Mumbai,Economy,9.92,21.0,5605
39972,39972,Indigo,0.00E+00,Mumbai,NaN,one,Evening,Kolkata,Economy,6.67,13.0,11088
39978,39978,Indigo,0.00E+00,Kolkata,Afternoon,one,Night,Delhi,Economy,7.08,NaN,5636


### handling discrepancy in flight route name that has this format '6.00E', "0.00E+00","0.00E-02"
- flight

impute 'NaN' if the flight number is not meaningful



In [26]:


def clean_flight_1(row):
    """
    correcting the flight number
    flight -- 6.00E-128 to 6E-128

    """

    route = row['flight']
    if (route[0:5] == "6.00E") :
        return'6E'+route[5:]
    else:
        return route



def clean_flight_2(row):
    """
    if the flight number is not meaningful then replace it with np.nan
    "0.00E+00","0.00E-02"  to np.nan

    """
    route = row['flight']
    if  (route in ["0.00E+00","0.00E-02"]):
        return np.nan
    else:
        return route


airline_flight={'I5':'AirAsia','AI':'Air_India','G8':'GO_FIRST','6E':'Indigo','SG':'SpiceJet','UK':'Vistara'}

def impute_airline(row):
    """
    if the airline name if missing we can guess it from the initial two digits of flight rought

    """
    if pd.isna(row['airline']):
        prefix = row['flight'][:2]
        if prefix in airline_flight:
            return airline_flight[prefix]
    return row['airline']




def logical_imputer(df):

    df['flight']=df.apply(clean_flight_1, axis=1)
    df['airline']=df.apply(impute_airline, axis=1)
    df['flight']=df.apply(clean_flight_2, axis=1)

    df['departure'] = df['departure'].where(
        df['departure'].notna(),df.groupby(['airline','source','destination','arrival']
                                           )['departure'].transform(
                                               lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x.bfill())))



    # df['duration'] = df['duration'].where(
    #     df['duration'].notna(),df.groupby(['airline','source', 'stops', 'destination']
    #                                       )['duration'].transform(
    #                                           lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x.median()[0])))


    df['stops'] = df['stops'].where(
        df['stops'].notna(),df.groupby(['airline','source','destination','duration']
                                       )['stops'].transform(
                                           lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x.bfill())))

    df['days_left'] = df['days_left'].where(
        df['days_left'].notna(),df.groupby(['source','arrival','destination','class','price']
                                       )['days_left'].transform(
                                           lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x.bfill())))



    df['flight'] = df['flight'].where(
        df['flight'].notna(),df.groupby(['airline','arrival','source','destination']
                                       )['flight'].transform(
                                           lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x.bfill())))



    for name, group_df in df.groupby(['airline','source', 'stops', 'destination'] ):

        for idx, row in group_df.iterrows():

            if pd.isna(row['duration']):

                candidates = (group_df[ group_df['duration'].notna()].drop(idx, errors='ignore'))

                if candidates.empty:
                    continue

                closest_index = ( candidates['price'] - row['price']).abs().idxmin()

                closest_value = candidates.loc[closest_index,'duration']

                df.loc[idx, 'duration'] = closest_value



    df.dropna(inplace=True)

    return df





In [27]:
df = pd.read_csv("/content/flight_data.csv")

df = logical_imputer(df)


print(df.isna().sum())

/tmp/ipykernel_1550/4001652483.py:54: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x.bfill())))
/tmp/ipykernel_1550/4001652483.py:67: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x.bfill())))
/tmp/ipykernel_1550/4001652483.py:79: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `p

id             0
airline        0
flight         0
source         0
departure      0
stops          0
arrival        0
destination    0
class          0
duration       0
days_left      0
price          0
dtype: int64


In [28]:
df

,id,airline,flight,source,departure,stops,arrival,destination,class,duration,days_left,price
0,0,Vistara,UK-930,Mumbai,Early_Morning,one,Night,Chennai,Business,12.75,40.0,64173
1,1,Air_India,AI-539,Chennai,Evening,one,Morning,Mumbai,Economy,16.08,26.0,4357
2,2,SpiceJet,SG-8107,Delhi,Early_Morning,zero,Morning,Chennai,Economy,2.92,25.0,3251
4,4,Air_India,AI-569,Chennai,Early_Morning,one,Morning,Bangalore,Economy,4.83,20.0,3584
5,5,AirAsia,I5-620,Mumbai,Early_Morning,one,Morning,Delhi,Economy,6.17,34.0,2336
...,...,...,...,...,...,...,...,...,...,...,...,...
39993,39993,Vistara,UK-870,Hyderabad,Night,one,Evening,Kolkata,Business,21.50,42.0,58394
39995,39995,Vistara,UK-940,Mumbai,Morning,one,Evening,Bangalore,Economy,21.25,43.0,6450
39996,39996,Vistara,UK-720,Kolkata,Early_Morning,one,Night,Mumbai,Business,14.08,12.0,64831
39998,39998,Vistara,UK-940,Mumbai,Early_Morning,one,Night,Bangalore,Business,24.67,4.0,84557
